In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from copy import deepcopy

from pathlib import Path

# sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))
# os.environ['PYTHONPATH'] = ':'.join(sys.path)

In [2]:
import awkward as ak
import numpy as np

from hepattn.experiments.atlas.performance.performance import PerformanceATLAS
from hepattn.experiments.atlas.performance.reader import load_truth_atlas

### Thresholds

In [3]:
IND_THRESHOLD = 0.50
ASSOC_TRACK_PT_THRESHOLD = 100

### Paths

In [4]:
# data sources
JZ_PARTITIONS_PATH = Path(os.environ.get("JZ_PATH_PARTITIONS", ""))

# EXP_NAME = "glow_baseline/atlas_run4_jz1234_v1"
# EXP_NAME = "glow_baseline/atlas_v1_nopart"
EXP_NAME = "jz1234_v0_nopart_reproduce"

# CKPT_PATH = JZ_PARTITIONS_PATH.parent.parent / f"results/{EXP_NAME}"
CKPT_PATH = JZ_PARTITIONS_PATH.parent.parent / f"results/{EXP_NAME}/atlas_jz1234_v0_nopart_reproduce_20260630-T221535/ckpts"

# PRED_NAME = "epoch=027-val_loss=5.33018__test_baseline.root"
# PRED_NAME = "epoch=199-val_loss=5.25962-43799__test_baseline.root"
PRED_NAME = "epoch=016-val_loss=5.32814__test_baseline.root"

# list of raw val/test filenames
truth_path = JZ_PARTITIONS_PATH / "val_raw_paths.txt"
truth_cache_path = None

# expr for merged jets reco (one per JZ dataset)
truth_jets_path = str(JZ_PARTITIONS_PATH.parent / "edreyer/merged_JZ*_jets.root")

# dictionary of precomputed prediction paths (one per model)
pred_paths = {
    "mpflow": CKPT_PATH / PRED_NAME,
}

RESULTS_PATH = Path(os.environ.get("HOME", Path("~").expanduser())) / "projects" / "hepattn" / "results" / EXP_NAME
save_dir = RESULTS_PATH / "plots_run4"
SAVE_FIG = True
if not Path(save_dir).exists():
    os.makedirs(save_dir)
FIGS = "plot"

print(f"Results path: {RESULTS_PATH}")
print(f"Precomputed predictions read from: {pred_paths}")
print(f"Truth data read from: {truth_path}")
print(f"Truth jets read from: {truth_jets_path}")


Results path: /home/lclissa/projects/hepattn/results/jz1234_v0_nopart_reproduce
Precomputed predictions read from: {'mpflow': PosixPath('/fast_scratch_4/atlas/pflow/0pileup_JZ1-4_Run4_AOD_samples/partitions/GLOW/results/jz1234_v0_nopart_reproduce/atlas_jz1234_v0_nopart_reproduce_20260630-T221535/ckpts/epoch=016-val_loss=5.32814__test_baseline.root')}
Truth data read from: /fast_scratch_4/atlas/pflow/0pileup_JZ1-4_Run4_AOD_samples/partitions/GLOW/data/dijet_JZ1234_June2026_2000/val_raw_paths.txt
Truth jets read from: /fast_scratch_4/atlas/pflow/0pileup_JZ1-4_Run4_AOD_samples/partitions/GLOW/data/edreyer/merged_JZ*_jets.root


### Styles

In [5]:
style_sheet = {
    'LABELS': {
        ### algorithms
        'truth': 'Truth',
        'ppflow': 'PPflow',
        'pandora': 'Pandora',
        'topo': 'TopoJet',
        'proxy': 'Proxy (track-sub)',
        'pflow': 'Proxy + Tracks',
        'hybrid': 'Proxy (calo-only)',
        'final': 'HGPflow Hybrid',
        'mpflow': 'GLOW',
        'hgpflow': 'HGPflow',
        'hgpflow_mini': 'HGPflow mini',
        'hgpflow_target': 'HGPflow target',
        'mlpf': 'MLPF',
        'empflow': 'EMPFlow',
        'emtopo': 'EMTopo',
        'lctopo': 'LCTopo',
        ### particles
        0: 'ch. had', 
        1: '$e^\pm$', 
        2: '$\mu^\pm$', 
        3: 'nu. had', 
        4: '$\gamma$', 
        5: 'resid.',
    },

    'LABEL_LEN': {
        'truth': 11,
        'proxy': 11,
        'pflow': 11,
        'hybrid': 10,
        'final': 10,
        'mpflow': 10,
        'topo' : 11,
        'ppflow': 10,
        'hgpflow': 8,
        'hgpflow_target': 8,
        'empflow': 10,
        'emtopo': 9,
    },

    'LINE_STYLES': {
        'truth': '--',
        'topo' : '-',
        'ppflow': '-',
        'proxy': '--',
        'hybrid': '--',
        'final': '-',
        'mpflow': '-',
        'pflow': '--',
        'hgpflow': '-',
        'hgpflow_target': '-',
        'empflow': '-',
        'emtopo': '--'
    },

    'MARKERS': {
        'truth': 'x',
        'topo': 'o',
        'ppflow': 'o',
        'proxy': 'x',
        'pflow': 'x',
        'hybrid': 'x',
        'final': '^',
        'hgpflow': 'o',
        'hgpflow_target': 'x',
        'mpflow': '^',
        'empflow': 'h',
        'emtopo': 's'
    },

    'COLORS': {
        'truth': 'black',
        'topo': 'greenyellow',
        'ppflow': 'gray',
        'proxy': 'blue',
        'pflow': 'purple',
        'hybrid': 'purple',
        'final': 'firebrick',
        'hgpflow': 'firebrick',
        'mpflow': 'darkorange',
        'hgpflow_target': 'darkorange',
        'empflow': 'teal',
        'emtopo': 'olivedrab'
    },

    'HISTTYPES': {
        'truth': 'step',
        'ppflow': 'bar',
        'pandora': 'bar',
        'topo': 'bar',
        'proxy': 'step',
        'pflow': 'step',
        'hybrid': 'step',
        'final': 'step',
        'mpflow': 'step',
        'hgpflow': 'step',
        'hgpflow_mini': 'step',
        'hgpflow_target': 'step',
        'mlpf': 'step',
        'empflow': 'bar',
        'emtopo': 'bar',
        'lctopo': 'bar',
    },

    'ALPHAS': {
        'truth': 1.0,
        'ppflow': 0.5,
        'pandora': 0.5,
        'topo': 0.5,
        'proxy': 1.0,
        'hybrid': 1.0,
        'final': 1.0,
        'mpflow': 1.0,
        'pflow': 1.0,
        'hgpflow': 1.0,
        'hgpflow_mini': 1.0,
        'hgpflow_target': 1.0,
        'mlpf': 1.0,
        'empflow': 0.5,
        'emtopo': 0.5,
        'lctopo': 0.5,
    }
}

<>:22: SyntaxWarning: invalid escape sequence '\p'
<>:23: SyntaxWarning: invalid escape sequence '\m'
<>:25: SyntaxWarning: invalid escape sequence '\g'
<>:22: SyntaxWarning: invalid escape sequence '\p'
<>:23: SyntaxWarning: invalid escape sequence '\m'
<>:25: SyntaxWarning: invalid escape sequence '\g'
/tmp/ipykernel_1006021/790348322.py:22: SyntaxWarning: invalid escape sequence '\p'
  1: '$e^\pm$',
/tmp/ipykernel_1006021/790348322.py:23: SyntaxWarning: invalid escape sequence '\m'
  2: '$\mu^\pm$',
/tmp/ipykernel_1006021/790348322.py:25: SyntaxWarning: invalid escape sequence '\g'
  4: '$\gamma$',


# Dijet

In [6]:
# note: truth_path must be passed as string to make it work
truth_dict = load_truth_atlas(str(truth_path), topo=False, fiducial_cuts=False, 
                              cache_path=truth_cache_path, jets_path=truth_jets_path)

Loading truth from cache: /fast_scratch_4/atlas/pflow/0pileup_JZ1-4_Run4_AOD_samples/partitions/GLOW/data/dijet_JZ1234_June2026_2000/val_raw_paths_topo0_fid0.parquet
Loading jets from 4 file(s) with 32 workers...


In [7]:
# JZ 123
# event_numbers_jz123 = np.arange(7500 * 3)

perf_obj = PerformanceATLAS(
    truth_path=truth_dict,
    pred_paths=pred_paths, 
    ind_threshold=IND_THRESHOLD, 
    topo=False, proxy=True, target_path=None, load_hung_matched_truth=False,
    fiducial_cuts_on_truth=False,
    # event_numbers=event_numbers_jz123,
    load_truth=True,
    entry_stop=None,
    num_workers=32)

Loading mpflow predictions: 95996/95996 events, 32 workers, step_size=50000


Loading mpflow predictions...: 100%|██████████| 2/2 [00:04<00:00,  2.48s/it]


  read done in 9.8s
  load_predictions total: 25.1s
common event count: 89498


Filtering and reordering mpflow predictions...: 100%|██████████| 19/19 [00:00<00:00, 2220.20it/s]


In [8]:
perf_obj.compute_jets(n_procs=45)

Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
AntiKt4TruthJets
AntiKt4EMPFlowJets
AntiKt4EMTopoJets
mpflow
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Jet clustering algorithm:  antikt
Jet clustering radius:  Processed 0/2088 events...
0.4
Jet clustering algorithm:  antikt
 Jet clustering radius: 0.4
Jet clustering algorithm:  antikt
Jet clustering radius:  Processed 0/2089 events...Processed 0/2089 events...
0.4

Jet clustering algorithm:  #--------------------------------------------------------------------------
#                         FastJet release 3.5.1
#                 M. Cacciari, G.P. Salam and G. Soyez                  
#     A software package for jet finding and analysis at colliders      
#                           https://fastjet.fr                           
#	                                                                      
# Please cite EPJC72(2012)1896 [arXiv:1111.6097] if you use this package
# for scientific work and optio

In [9]:
perf_obj.match_jets()

Matching jets...: 100%|██████████| 93998/93998 [00:08<00:00, 10899.90it/s]


In [10]:
def get_track_substituted_perf(is_charged_mask, ptetaphi, trackptetaphi, tag=None, track_pt_threshold=ASSOC_TRACK_PT_THRESHOLD):
    out_dict = {}

    if tag is not None:
        tag = f'{tag}_'

    # eta-phi for all charged particles
    out_dict[tag + 'eta'] = ak.where(
        is_charged_mask, 
        trackptetaphi['eta'], 
        ptetaphi['eta'])

    out_dict[tag + 'phi'] = ak.where(
        is_charged_mask, 
        trackptetaphi['phi'], 
        ptetaphi['phi'])

    # pt only below certain track pT. Above threshold, use default pt (can be from mpflow or calo proxy)
    assoc_track_pt_mask = trackptetaphi['pt'] < track_pt_threshold
    ch_pt_mask = is_charged_mask * assoc_track_pt_mask

    out_dict[tag + 'pt'] = ak.where(
        ch_pt_mask, 
        trackptetaphi['pt'], 
        ptetaphi['pt'])
    
    return out_dict

In [11]:
if 'hgpflow' in pred_paths:
    hgpflow_class = perf_obj.pred_dicts['hgpflow']['hgpflow_class']
    hgpflow_is_charged = hgpflow_class <= 2
    hgpflow_ptetaphi = {
        'pt': perf_obj.pred_dicts['hgpflow']['hgpflow_pt'],
        'eta': perf_obj.pred_dicts['hgpflow']['hgpflow_eta'],
        'phi': perf_obj.pred_dicts['hgpflow']['hgpflow_phi']
    }
    proxy_ptetaphi = {
        'pt': perf_obj.pred_dicts['hgpflow']['proxy_pt'],
        'eta': perf_obj.pred_dicts['hgpflow']['proxy_eta'],
        'phi': perf_obj.pred_dicts['hgpflow']['proxy_phi']
    }
    track_ptetaphi = {
        'pt': perf_obj.pred_dicts['hgpflow']['assoc_track_pt'],
        'eta': perf_obj.pred_dicts['hgpflow']['assoc_track_eta'],
        'phi': perf_obj.pred_dicts['hgpflow']['assoc_track_phi']
    }
    hgpflow_track_sub_dict = get_track_substituted_perf(hgpflow_is_charged,
                                                        hgpflow_ptetaphi, 
                                                        track_ptetaphi, 
                                                        tag='hgpflow_tracksub',
                                                        track_pt_threshold=ASSOC_TRACK_PT_THRESHOLD)
    
    hgpflow_proxy_track_sub_dict = get_track_substituted_perf(hgpflow_is_charged,
                                                        proxy_ptetaphi, 
                                                        track_ptetaphi, 
                                                        tag='hgpflow_proxy_tracksub',
                                                        track_pt_threshold=ASSOC_TRACK_PT_THRESHOLD)

    perf_obj.pred_dicts['hgpflow'].update({k: v for k, v in hgpflow_track_sub_dict.items()})
    perf_obj.pred_dicts['hgpflow'].update({k: v for k, v in hgpflow_proxy_track_sub_dict.items()})

if 'mpflow' in pred_paths:

    mpflow_proxy_is_charged = perf_obj.pred_dicts['mpflow']['proxy_is_charged']

    track_ptetaphi = {
        'pt': perf_obj.pred_dicts['mpflow']['proxy_ch_pt'],
        'eta': perf_obj.pred_dicts['mpflow']['proxy_ch_eta'],
        'phi': perf_obj.pred_dicts['mpflow']['proxy_ch_phi']
    }

    proxy_ptetaphi = {
        'pt': perf_obj.pred_dicts['mpflow']['proxy_pt'],
        'eta': perf_obj.pred_dicts['mpflow']['proxy_eta'],
        'phi': perf_obj.pred_dicts['mpflow']['proxy_phi']
    }

    mpflow_ptetaphi = {
        'pt': perf_obj.pred_dicts['mpflow']['mpflow_pt'],
        'eta': perf_obj.pred_dicts['mpflow']['mpflow_eta'],
        'phi': perf_obj.pred_dicts['mpflow']['mpflow_phi']
    }

    mpflow_track_sub_dict = get_track_substituted_perf(mpflow_proxy_is_charged,
                                                        mpflow_ptetaphi, 
                                                        track_ptetaphi,
                                                        tag='mpflow_tracksub',
                                                        track_pt_threshold=ASSOC_TRACK_PT_THRESHOLD)
    
    mpflow_proxy_track_sub_dict = get_track_substituted_perf(mpflow_proxy_is_charged,
                                                        proxy_ptetaphi, 
                                                        track_ptetaphi,
                                                        tag='mpflow_proxy_tracksub',
                                                        track_pt_threshold=ASSOC_TRACK_PT_THRESHOLD)

    ### Additional collection: mpflow_track_sub but with raw calo proxy substituted for charged particles above track pT threshold
    ### (i.e. substitute tracks for low-pT charged particles, substitute raw calo for high-pT charged particles)
    hybrid_ptetaphi = {
        'pt': ak.where(mpflow_proxy_is_charged,  
                       perf_obj.pred_dicts['mpflow']['proxy_neut_pt'], 
                       perf_obj.pred_dicts['mpflow']['mpflow_pt']),
        'eta': perf_obj.pred_dicts['mpflow']['mpflow_eta'],
        'phi': perf_obj.pred_dicts['mpflow']['mpflow_phi']
    }
    mpflow_hybrid_track_sub_dict = get_track_substituted_perf(mpflow_proxy_is_charged,
                                                        hybrid_ptetaphi, 
                                                        track_ptetaphi,
                                                        tag='mpflow_tracksub_calosub',
                                                        track_pt_threshold=ASSOC_TRACK_PT_THRESHOLD)
    
    perf_obj.pred_dicts['mpflow'].update({k: v for k, v in mpflow_track_sub_dict.items()})
    perf_obj.pred_dicts['mpflow'].update({k: v for k, v in mpflow_proxy_track_sub_dict.items()})
    perf_obj.pred_dicts['mpflow'].update({k: v for k, v in mpflow_hybrid_track_sub_dict.items()})


In [ ]:
from hepattn.experiments.atlas.performance.jet_helper import JetHelper, compute_jets
jet_obj = JetHelper(radius=0.4, algo='antikt')

model_name = 'mpflow'
pred_dict = perf_obj.pred_dicts[model_name]

for key in ['tracksub', 'proxy_tracksub', 'tracksub_calosub']:
    actual_key = f'{model_name}_{key}'
    print(f'{model_name} {key}')
    pred_dict[key + '_jets'] = compute_jets(jet_obj,
        pred_dict[f'{actual_key}_pt'], pred_dict[f'{actual_key}_eta'],
        pred_dict[f'{actual_key}_phi'], pred_dict[f'{model_name}_mass'],
        fourth_name='mass', n_procs=30)

Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
mpflow tracksub


Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Processed 0/3133 events...
Processed 0/3133 events...
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Processed 0/3133 events...
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Jet clustering algorithm: 
 antiktJet clustering radius: #--------------------------------------------------------------------------
#                         FastJet release 3.5.1
#                 M. Cacciari, G.P. Salam and G. Soyez                  
#     A software package for jet finding and analysis at colliders      
#                           https://fastjet.fr                           
#	                                                                      
# Please cite EPJC72(2012)1896 [arXiv:1111.6097] if you use this package
# for scientific work and optionally PLB641(2006)57 [hep-ph/0512210].   
#                                                  

Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Jet clustering algorithm:  antiktProcessed 0/3133 events...

Jet clustering radius:  0.4
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Processed 0/3133 events...
Processed 0/3133 events...
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Jet clustering algorithm: 
 Processed 0/3134 events...antikt

Jet clustering radius:  0.4Processed 0/3133 events...
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Processed 0/3133 events...
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Processed 0/3133 events...
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Processed 0/3134 events...
Jet clustering algorithm:  antikt
Jet clustering radius:  0.4
Processed 0/3133 events...
Jet clustering algorithm: Jet clustering radius:  antikt
 0.4
Processed 0/3133 events...
Jet clustering algorithm: 
 antikt
Jet clustering radius:  0.4Processed 0/3133 events...
Jet clustering algori

In [13]:
### Calo proxy mpflow jets
pred_dict['mpflow_calo_jets'] = compute_jets(jet_obj,
    pred_dict['proxy_neut_pt'],
    ak.where(pred_dict['proxy_is_charged'], pred_dict['proxy_ch_eta'], pred_dict['proxy_neut_eta']),
    ak.where(pred_dict['proxy_is_charged'], pred_dict['proxy_ch_phi'], pred_dict['proxy_neut_phi']),
    pred_dict['mpflow_mass'],
    fourth_name='mass', n_procs=30)

In [14]:
### match additional collections

model_name = 'mpflow'
pred_dict = perf_obj.pred_dicts[model_name]
for key in ['tracksub', 'proxy_tracksub', 'tracksub_calosub', 'mpflow_calo']:
    print(f'{model_name} {key}')
    pred_dict[f'matched_{key}_jets'] = perf_obj.match_jets_all_ev(
        perf_obj.truth_dict['AntiKt4TruthJets'],
        pred_dict[key + '_jets']
    )

mpflow tracksub


Matching jets...: 100%|██████████| 93998/93998 [00:09<00:00, 9530.67it/s] 


mpflow proxy_tracksub


Matching jets...: 100%|██████████| 93998/93998 [00:08<00:00, 11005.04it/s]


mpflow tracksub_calosub


Matching jets...: 100%|██████████| 93998/93998 [00:08<00:00, 11033.04it/s]


mpflow mpflow_calo


Matching jets...: 100%|██████████| 93998/93998 [00:07<00:00, 12084.59it/s]


### Jet residual plots

In [15]:
def find_matching_jet(jets, target_jet, delta_r_threshold=0.1):
    '''
    Find the jet in 'jets' that matches the target_eta and target_phi within delta_r_threshold.
    Returns the index of the matching jet, or -1 if no match is found.
    '''
    delta_rs = []
    for idx, jet in enumerate(jets):
        delta_r = jet.delta_R(target_jet)
        delta_rs.append((delta_r, idx))
    delta_rs.sort(key=lambda x: x[0])  # Sort by delta_r
    if delta_rs and delta_rs[0][0] < delta_r_threshold:
        return delta_rs[0][1]
    else:
        return -1

def decaying_coeff(y, y_threshold, decay_ratio=0.5):
    decay_rate = decay_ratio * y_threshold
    erf = lambda x: (1 + np.tanh(x / decay_rate)) / 2
    alpha = erf(y_threshold - y)
    return alpha


from copy import deepcopy
def best_of_both_jets(low_pt_matched_jets, high_pt_matched_jets, comp_jet_pt_threshold=250, delta_r_threshold=0.1, pt_consistency_threshold=None):
    '''
    Combine two sets of matched jets, taking jets from the low_pt set if their pt is below a threshold,
    and from the high_pt set if their pt is above the threshold.
    '''
    low_ref_jets, low_comp_jets = low_pt_matched_jets
    high_ref_jets, high_comp_jets = high_pt_matched_jets
    best_ref_jets, best_comp_jets = [], []
    for ev in range(len(low_ref_jets)):
        ref_jets_ev = []
        comp_jets_ev = []
        for low_ref_jet, low_comp_jet in zip(low_ref_jets[ev], low_comp_jets[ev]):
            substitute_comp_jet = None
            if (low_comp_jet.pt - comp_jet_pt_threshold)/comp_jet_pt_threshold > -0.5:
                ### look for matching comp jet in high pt jets
                match_idx = find_matching_jet(high_comp_jets[ev], low_comp_jet, delta_r_threshold=delta_r_threshold)
                if match_idx >= 0:
                    
                    substitute_comp_jet = deepcopy(high_comp_jets[ev][match_idx])

                    if pt_consistency_threshold is not None:
                        ### Final check: pt consistency near threshold
                        if (low_comp_jet.pt - comp_jet_pt_threshold)/comp_jet_pt_threshold < pt_consistency_threshold:
                            relative_pt_diff = np.abs(substitute_comp_jet.pt - low_comp_jet.pt) / low_comp_jet.pt
                            if relative_pt_diff > pt_consistency_threshold:
                                substitute_comp_jet = None


            if substitute_comp_jet is None:
                comp_jets_ev.append(low_comp_jet)
            else:
                ### Smooth average
                alpha = decaying_coeff(low_comp_jet.pt, y_threshold=comp_jet_pt_threshold)
                substitute_comp_jet.pt = alpha * low_comp_jet.pt + (1 - alpha) * substitute_comp_jet.pt
                comp_jets_ev.append(substitute_comp_jet)

            ### will not "cheat" -- need to keep ref jet the same
            ref_jets_ev.append(low_ref_jet)

        best_ref_jets.append(ref_jets_ev)
        best_comp_jets.append(comp_jets_ev)
    return best_ref_jets, best_comp_jets

In [16]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
thresh = 400
y = np.linspace(10, 1000, 100)

ax.plot(y, decaying_coeff(y, y_threshold=thresh))
ax.plot([thresh], [decaying_coeff(thresh, y_threshold=thresh)], 'ro')
ax.set_xlabel('y')
ax.set_ylabel('decaying coeff')
ax.set_title('Decaying coefficient as a function of y')
plt.grid()

In [17]:
if 'hgpflow' in pred_paths:
    hybrid_hgpflow_proxy_jets = best_of_both_jets(
        perf_obj.pred_dicts['hgpflow']['matched_proxy_tracksub_jets'],
        perf_obj.pred_dicts['hgpflow']['matched_proxy_jets'],
        comp_jet_pt_threshold=400,
        delta_r_threshold=0.4,
        # pt_consistency_threshold=0.2
    )

    hybrid_hgpflow_jets = best_of_both_jets(
        perf_obj.pred_dicts['hgpflow']['matched_tracksub_jets'],
        perf_obj.pred_dicts['hgpflow']['matched_proxy_jets'],
        comp_jet_pt_threshold=400,
        delta_r_threshold=0.4,
        # pt_consistency_threshold=0.1
    )

if 'mpflow' in pred_paths:
    hybrid_mpflow_proxy_jets = best_of_both_jets(
        perf_obj.pred_dicts['mpflow']['matched_proxy_tracksub_jets'],
        perf_obj.pred_dicts['mpflow']['matched_mpflow_calo_jets'],
        comp_jet_pt_threshold=400,
        delta_r_threshold=0.4,
        # pt_consistency_threshold=0.1
    )

    hybrid_mpflow_jets = best_of_both_jets(
        perf_obj.pred_dicts['mpflow']['matched_tracksub_jets'],
        perf_obj.pred_dicts['mpflow']['matched_mpflow_calo_jets'],
        comp_jet_pt_threshold=400,
        delta_r_threshold=0.4,
        # pt_consistency_threshold=0.1
    )


In [18]:
for k in perf_obj.pred_dicts['mpflow'].keys():
    if 'matched' in k:
        print(k)

matched_mpflow_jets
matched_proxy_jets
matched_tracksub_jets
matched_proxy_tracksub_jets
matched_tracksub_calosub_jets
matched_mpflow_calo_jets


In [19]:
from hepattn.experiments.atlas.performance.plot_helper_event import compute_jet_residual_dict

_dict = {
    'emtopo' : perf_obj.truth_dict['matched_AntiKt4EMTopoJets'],
    'empflow': perf_obj.truth_dict['matched_AntiKt4EMPFlowJets'],
    # 'proxy'  : perf_obj.pred_dicts['hgpflow']['matched_proxy_jets'],
    # 'proxy'  : perf_obj.pred_dicts['hgpflow']['matched_proxy_tracksub_jets'],
    # 'hybrid': hybrid_proxy_jets,
    # 'hgpflow': perf_obj.pred_dicts['hgpflow']['matched_hgpflow_jets'],
    # 'hgpflow': perf_obj.pred_dicts['hgpflow']['matched_tracksub_jets'],
    # 'final': hybrid_hgpflow_jets,
    # 'proxy'  : perf_obj.pred_dicts['mpflow']['matched_proxy_jets'],
    'mpflow': perf_obj.pred_dicts['mpflow']['matched_mpflow_jets'],
    'proxy': perf_obj.pred_dicts['mpflow']['matched_proxy_tracksub_jets'],
    # 'mpflow': perf_obj.pred_dicts['mpflow']['matched_tracksub_jets'], #tracksub_calosub_jets'],
    'hybrid': perf_obj.pred_dicts['mpflow']['matched_mpflow_calo_jets'],
    # 'hybrid': hybrid_mpflow_jets, #perf_obj.pred_dicts['mpflow']['matched_proxy_tracksub_jets'],
    # 'pflow'  : perf_obj.pred_dict['matched_pflow_jets'],
    # 'topo'   : perf_obj.truth_dict['matched_topo_jets'],
    # 'truth': perf_obj.truth_dict['matched_truth_jets'],
    # f'{model_name}_target': perf_obj.hgpflow_target_dict['matched_hgpflow_target_jets'],
}

jet_residual_dict = compute_jet_residual_dict(_dict, dr_cut=0.1, leading_N_jets=2, pt_min=20)

In [20]:
from hepattn.experiments.atlas.performance.plot_helper_event import plot_jet_residuals

figs = plot_jet_residuals(jet_residual_dict, pt_relative=True, stylesheet=style_sheet, separate_figures=False)
if SAVE_FIG:
    # figs[0].savefig(f'{save_dir}/dijet_jet_residuals_pt.png', dpi=300)
    # figs[1].savefig(f'{save_dir}/dijet_jet_residuals_dr.png', dpi=300)
    # figs[2].savefig(f'{save_dir}/dijet_jet_residuals_constcount.png', dpi=300)
    figs.savefig(f'{save_dir}/{FIGS}_dijet_jet_residuals.png', dpi=300)

In [21]:
from hepattn.experiments.atlas.performance.plot_helper_event import plot_jet_ratio_boxplot

pt_bins = np.array([20, 50, 100, 150, 200, 250, 300, 350, 400, 450])
# pt_bins = np.array([20, 50, 100, 200, 300, 400, 600, 800, 1000])
# pt_bins = np.array([20, 50, 100, 200, 300, 400, 600, 800, 1000, 1500])

fig_pt = plot_jet_ratio_boxplot(jet_residual_dict, bins=pt_bins, stylesheet=style_sheet)

# eta_bins = np.linspace(-3, 3, 15)
abs_eta_bins = np.linspace(0, 3, 6)
fig_eta = plot_jet_ratio_boxplot(jet_residual_dict, var='abseta', bins=abs_eta_bins, stylesheet=style_sheet)

if SAVE_FIG:
    fig_pt.savefig(f'{save_dir}/{FIGS}_dijet_jet_ratio_pt_boxplot.png', dpi=300)
    fig_eta.savefig(f'{save_dir}/{FIGS}_dijet_jet_ratio_eta_boxplot.png', dpi=300)

In [22]:
from hepattn.experiments.atlas.performance.plot_helper_event import plot_jet_response

pt_bins = np.array([20, 50, 100, 150, 200, 250, 300, 350, 400, 450])
# pt_bins = np.array([20, 50, 100, 200, 300, 400, 600, 800, 1000])
# pt_bins = np.array([20, 50, 100, 200, 300, 400, 600, 800, 1000, 1500])
figs = plot_jet_response(jet_residual_dict, pt_bins=pt_bins, separate_figures=True, stylesheet=style_sheet, ratio_panel=True)

if SAVE_FIG:
    figs[0].savefig(f'{save_dir}/{FIGS}_dijet_jet_response_meanstd.png', dpi=300)
    figs[1].savefig(f'{save_dir}/{FIGS}_dijet_jet_response_medianiqr.png', dpi=300)

#### Marginal distributions

In [23]:
_jet_dict = {
    # 'truth': perf_obj.truth_dict['truth_jets'],
    'truth': perf_obj.truth_dict['AntiKt4TruthJets'],
    'empflow': perf_obj.truth_dict['AntiKt4EMPFlowJets'],
    'emtopo': perf_obj.truth_dict['AntiKt4EMTopoJets'],
    # 'hgpflow': perf_obj.pred_dicts['hgpflow']['jets'],
    # 'proxy': perf_obj.pred_dicts['mpflow']['jets'],
    'mpflow': perf_obj.pred_dicts['mpflow']['jets'],
    # 'topo': perf_obj.truth_dict['topo_jets'],
}

from hepattn.experiments.atlas.performance.plot_helper_event import plot_jet_marginals
fig = plot_jet_marginals(_jet_dict, nleading=2, stylesheet=style_sheet)

#### Efficiency and fakerate

In [24]:
perf_obj.hung_match_particles(flatten=True, return_unmatched=True, dR_threshold=0.5)

ak->np comversion... done


Matching particles...: 100%|██████████| 93998/93998 [08:58<00:00, 174.64it/s]


ak->np comversion... done


Matching particles...: 100%|██████████| 93998/93998 [10:03<00:00, 155.80it/s]


In [25]:
style_sheet_eff_fr = {
    'LABELS': {
        'hgpflow': 'HGPflow',
        'mpflow': 'GLOW',
    },
    'LINE_STYLES': {
        'hgpflow': '-',
        'mpflow': '--',
    },
    'COLORS': {
        'hgpflow': {
            'neut had': 'mediumseagreen',
            'photon': 'tomato',
        },
        'mpflow': {
            'neut had': 'mediumseagreen',
            'photon': 'tomato',
        }
    }
}

In [26]:
perf_obj.pred_dicts['mpflow']['matched_mpflow_particles']

({'pt': array([1.47773826, 8.85765266, 8.38477612, ..., 1.70240045, 0.28995791,
         0.94164181], shape=(18079577,)),
  'eta': array([-1.43499756, -1.44493783, -1.66372609, ...,  2.86003375,
          2.79541326,  2.69180584], shape=(18079577,)),
  'phi': array([-0.42556018, -0.42179409, -0.34687009, ..., -3.00208807,
          2.56651616, -2.24602914], shape=(18079577,)),
  'class': array([0, 0, 0, ..., 4, 4, 4], shape=(18079577,))},
 {'pt': array([1.37549806, 9.28861141, 8.13629627, ..., 1.66409254, 0.25886339,
         0.49574155], shape=(18079577,)),
  'eta': array([-1.43554688, -1.453125  , -1.6640625 , ...,  2.8359375 ,
          2.63671875,  2.765625  ], shape=(18079577,)),
  'phi': array([-0.42695081, -0.42178729, -0.34643689, ...,  3.12633944,
          2.60387588, -1.82145953], shape=(18079577,)),
  'class': array([0, 0, 0, ..., 3, 4, 3], shape=(18079577,))},
 {'pt': array([0.39985493, 1.03054464, 1.33006728, ..., 0.12602226, 0.09120487,
         0.12445994], shape=(54968

In [27]:
from hepattn.experiments.atlas.performance.plot_helper_particle import plot_eff_fr_purity

eff_fr_purity_input_dict = {
    # 'hgpflow': {
    #     'ref_matched': perf_obj.pred_dicts['hgpflow']['matched_hgpflow_particles'][0],
    #     'comp_matched': perf_obj.pred_dicts['hgpflow']['matched_hgpflow_particles'][1],
    #     'ref_unmatched': perf_obj.pred_dicts['hgpflow']['matched_hgpflow_particles'][2],
    #     'comp_unmatched': perf_obj.pred_dicts['hgpflow']['matched_hgpflow_particles'][3],
    # },
    'mpflow': {
        'ref_matched': perf_obj.pred_dicts['mpflow']['matched_mpflow_particles'][0],
        'comp_matched': perf_obj.pred_dicts['mpflow']['matched_mpflow_particles'][1],
        'ref_unmatched': perf_obj.pred_dicts['mpflow']['matched_mpflow_particles'][2],
        'comp_unmatched': perf_obj.pred_dicts['mpflow']['matched_mpflow_particles'][3],
    }
}

fig = plot_eff_fr_purity(eff_fr_purity_input_dict, stylesheet=style_sheet_eff_fr)
if SAVE_FIG:
    fig.savefig(f'{save_dir}/dijet_eff_fr_purity.png', dpi=300, bbox_inches='tight')

#### Residuals

In [28]:
from hepattn.experiments.atlas.performance.plot_helper_particle import plot_residuals

style_sheet_part_res = deepcopy(style_sheet)
style_sheet_part_res['COLORS']['proxy'] = 'dodgerblue'
style_sheet_part_res['LINE_STYLES']['proxy'] = '--'

_dict = {
    'mpflow': perf_obj.pred_dicts['mpflow']['matched_mpflow_particles'],
    # 'hgpflow': perf_obj.pred_dicts['hgpflow']['matched_hgpflow_particles'],
    'proxy': perf_obj.pred_dicts['mpflow']['matched_proxy_particles'],
}

qs = {
    'Charged': {'pt': 90, 'eta': 80, 'phi': 80}, 
    'Neutral': {'pt': 90, 'eta': 80, 'phi': 80}
}
fig = plot_residuals(_dict, pt_relative=True, log_y=False, qs=qs, stylesheet=style_sheet_part_res)
if SAVE_FIG:
    fig.savefig(f'{save_dir}/dijet_particle_residuals.png', dpi=300, bbox_inches='tight')

In [29]:
from hepattn.experiments.atlas.performance.plot_helper_particle import plot_residuals_neutrals

_dict = {
    'mpflow': perf_obj.pred_dicts['mpflow']['matched_mpflow_particles'],
    # 'hgpflow': perf_obj.pred_dicts['hgpflow']['matched_hgpflow_particles'],
    'proxy': perf_obj.pred_dicts['mpflow']['matched_proxy_particles'],
}

qs = {
    'Neutral hadron': {'pt': 98, 'eta': 75, 'phi': 75}, 
    'Photon': {'pt': 99, 'eta': 90, 'phi': 90}
}
fig = plot_residuals_neutrals(_dict, pt_relative=True, log_y=False, qs=qs, stylesheet=style_sheet_part_res, separate_figures=False)
if SAVE_FIG:
    fig.savefig(f'{save_dir}/dijet_particle_residuals_combined.png', dpi=300, bbox_inches='tight')
    # figs[0].savefig(f'{save_dir}/dijet_particle_residuals_neutralhad_pt.png', dpi=300, bbox_inches='tight')
    # figs[1].savefig(f'{save_dir}/dijet_particle_residuals_neutralhad_eta.png', dpi=300, bbox_inches='tight')
    # figs[2].savefig(f'{save_dir}/dijet_particle_residuals_neutralhad_phi.png', dpi=300, bbox_inches='tight')
    # figs[3].savefig(f'{save_dir}/dijet_particle_residuals_photon_pt.png', dpi=300, bbox_inches='tight')
    # figs[4].savefig(f'{save_dir}/dijet_particle_residuals_photon_eta.png', dpi=300, bbox_inches='tight')
    # figs[5].savefig(f'{save_dir}/dijet_particle_residuals_photon_phi.png', dpi=300, bbox_inches='tight')